# Phase 14.2 — Document Chunking

This notebook converts business documents into smaller chunks.

Chunking is required because the entire document should not be
sent to an LLM for every question.

Each chunk receives:

- chunk_id
- document_id
- file_name
- document_type
- chunk_index
- chunk_text

The chunks become the retrieval units for RAG.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    TimestampType
)

from datetime import datetime

In [0]:
DOCUMENT_TABLE = "genai_copilot.gold.business_documents"

CHUNK_TABLE = "genai_copilot.gold.business_document_chunks"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

print("Document table:", DOCUMENT_TABLE)
print("Chunk table:", CHUNK_TABLE)
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)

In [0]:
documents_df = spark.table(
    DOCUMENT_TABLE
)

display(
    documents_df.select(
        "document_id",
        "file_name",
        "title"
    )
)

In [0]:
def chunk_text(
    text,
    chunk_size=500,
    overlap=100
):
    """
    Split text into overlapping character-based chunks.

    This simple implementation is intentionally lightweight
    and suitable for the Free Edition portfolio project.
    """

    if text is None:
        return []

    text = text.strip()

    if not text:
        return []

    chunks = []

    start = 0
    chunk_index = 0

    while start < len(text):

        end = min(
            start + chunk_size,
            len(text)
        )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(
                (
                    chunk_index,
                    chunk
                )
            )

        if end >= len(text):
            break

        start = end - overlap

        chunk_index += 1

    return chunks

In [0]:
chunk_rows = []

for row in documents_df.collect():

    chunks = chunk_text(
        row["content"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk_index, chunk in chunks:

        chunk_id = (
            f"{row['document_id']}_"
            f"CHUNK_{chunk_index:04d}"
        )

        chunk_rows.append(
            (
                chunk_id,
                row["document_id"],
                row["file_name"],
                row["document_type"],
                row["title"],
                chunk_index,
                chunk,
                datetime.now()
            )
        )

print("Total chunks:", len(chunk_rows))

In [0]:
chunk_schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("document_type", StringType(), False),
    StructField("title", StringType(), False),
    StructField("chunk_index", IntegerType(), False),
    StructField("chunk_text", StringType(), False),
    StructField("created_at", TimestampType(), False)
])

chunks_df = spark.createDataFrame(
    chunk_rows,
    schema=chunk_schema
)

display(chunks_df)

In [0]:
chunks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(CHUNK_TABLE)

print("Chunk table created:")
print(CHUNK_TABLE)

In [0]:
print(
    "Documents:",
    spark.table(DOCUMENT_TABLE).count()
)

print(
    "Chunks:",
    spark.table(CHUNK_TABLE).count()
)

In [0]:
display(
    spark.table(CHUNK_TABLE)
    .select(
        "chunk_id",
        "file_name",
        "chunk_index",
        "chunk_text"
    )
    .orderBy(
        "document_id",
        "chunk_index"
    )
)


For this portfolio project, I think i don't need a complex document-processing framework yet.

We're demonstrating:

Document
   ↓
Chunking
   ↓
Metadata
   ↓
Retrieval

The production upgrade could later use:

PDF parsing
semantic chunking
token-based chunking
document sections
page metadata
table extraction

But the architectural concept remains the same.

We're not making an external vector database mandatory.

Databricks Free Edition currently supports an AI Search endpoint with one search unit, but Direct Vector Access isn't supported.

In [0]:
%sql
SELECT COUNT(*)
FROM genai_copilot.gold.business_documents;

In [0]:
%sql
SELECT
    document_id,
    file_name,
    COUNT(*) AS chunk_count
FROM genai_copilot.gold.business_document_chunks
GROUP BY
    document_id,
    file_name
ORDER BY document_id;